<a href="https://colab.research.google.com/github/NicolasLugo/Google-Colab/blob/main/Act_2_2_Nicol%C3%A1sLugo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Actividad 2.2. Reglas Rudimentarias 1R</h1>
<p>En este laboratorio construirás un modelo de clasificación con el algoritmo 1R (One Rule), primero en Orange y luego en Jupyter Notebook, para responder una pregunta de negocio con datos reales de una campaña.</p>

In [1]:
import pandas as pd
# Con el fin de practicar, primero haré el ejercicio con el CSV pequeño de prueba
df_clima = pd.read_csv('/content/datos/clima_jugar.csv')
display(df_clima)

,cielo,temperatura,humedad,viento,jugar
0,Soleado,Calor,Alta,No,No
1,Soleado,Calor,Alta,Sí,No
2,Nublado,Calor,Alta,No,Sí
3,Lluvioso,Templado,Alta,No,Sí
4,Lluvioso,Frío,Normal,No,Sí
5,Lluvioso,Frío,Normal,Sí,No
6,Nublado,Frío,Normal,Sí,Sí
7,Soleado,Templado,Alta,No,No
8,Soleado,Frío,Normal,No,Sí
9,Lluvioso,Templado,Normal,No,Sí


In [2]:
# Se debe definir la función que calcua el algoritmo 1R
def one_r(df, atributo, objetivo):
  # Se crea la tabla de frecuencias cruzada entre el atributo y la clase objetivo
  tabla = pd.crosstab(df[atributo], df[objetivo])
  # Se obtiene la clase mayoritaria para cada valor del atributo
  clase_mayoritaria = tabla.idxmax(axis=1)
  # Se calcula la cantidad total de errores (total menos aciertos de la clase mayoritaria)
  total_por_valor = tabla.sum(axis=1)
  aciertos = tabla.max(axis=1)
  errores = total_por_valor - aciertos

  print(f"--- Análisis para el atributo: {atributo} ---")
  for val in tabla.index:
    print(f"{val} -> {clase_mayoritaria[val]} (Errores: {errores[val]})")
  print(f"Errores totales: {errores.sum()}\n")

# Se prueba funcionalidad
one_r(df_clima, 'cielo', 'jugar')
one_r(df_clima, 'temperatura', 'jugar')
one_r(df_clima, 'humedad', 'jugar')
one_r(df_clima, 'viento', 'jugar')

--- Análisis para el atributo: cielo ---
Lluvioso -> Sí (Errores: 2)
Nublado -> Sí (Errores: 0)
Soleado -> No (Errores: 2)
Errores totales: 4

--- Análisis para el atributo: temperatura ---
Calor -> No (Errores: 2)
Frío -> Sí (Errores: 1)
Templado -> Sí (Errores: 2)
Errores totales: 5

--- Análisis para el atributo: humedad ---
Alta -> No (Errores: 3)
Normal -> Sí (Errores: 1)
Errores totales: 4

--- Análisis para el atributo: viento ---
No -> Sí (Errores: 2)
Sí -> No (Errores: 3)
Errores totales: 5



<h3>Resultados</h3>
<p>Viendo la cantidad de errores que tiene cada uno de los valores, está claro que se debe escoger como variable ganadora a la humedad, ya que al tener menos valores o categorías hace que la regla sea más simple y generalizable</p>

____________________________________________________________________________________________________________________________________________

<h3>Resolución caso Comercial Limarí</h3>
<p>Ahora, debemos aplicar una lógica parecida a la del ejercicio anterior, solo que ahora usando el archivo del caso con más variables y muchos más datos.</p>

In [3]:
# 1. Se debe cargar el dataset de clientes
df_clientes = pd.read_csv('/content/datos/clientes_campana.csv')

# 2. Luego, se revisan los tipos de datos y valores nulos
print(df_clientes.info())

# 3. Por último, se tratan los valores faltantes (imputación básica)
df_clientes['edad'] = df_clientes['edad'].fillna(df_clientes['edad'].median())
df_clientes['canal'] = df_clientes['canal'].fillna('Desconocido')

print("\nValores nulos restantes:", df_clientes.isnull().sum().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_cliente          400 non-null    object 
 1   comuna              400 non-null    object 
 2   edad                378 non-null    float64
 3   canal               385 non-null    object 
 4   tipo_cliente        400 non-null    object 
 5   tramo_gasto         400 non-null    object 
 6   visitas_ultimo_mes  400 non-null    int64  
 7   recibio_descuento   400 non-null    object 
 8   respondio           400 non-null    object 
dtypes: float64(1), int64(1), object(7)
memory usage: 28.3+ KB
None

Valores nulos restantes: 0


In [4]:
# Discretizar variables numéricas usando pandas cut
df_clientes['edad_tramo'] = pd.cut(df_clientes['edad'], bins=[0, 29.5, 44.5, 59.5, 100], labels=['Joven', 'Adulto Joven', 'Adulto', 'Senior'])
df_clientes['visitas_tramo'] = pd.cut(df_clientes['visitas_ultimo_mes'], bins=[-1, 1.5, 4.5, 100], labels=['Bajas', 'Medias', 'Altas'])

# Probar la función one_r en los atributos categóricos de los clientes (incluyendo los nuevos tramos)
for col in ['comuna', 'canal', 'tipo_cliente', 'tramo_gasto', 'recibio_descuento', 'edad_tramo', 'visitas_tramo']:
    one_r(df_clientes, col, 'respondio')

--- Análisis para el atributo: comuna ---
Combarbalá -> Sí (Errores: 26)
MONTE PATRIA -> Sí (Errores: 0)
Monte Patria -> Sí (Errores: 39)
OVALLE -> No (Errores: 1)
Ovalle -> No (Errores: 72)
PUNITAQUI -> No (Errores: 0)
Punitaqui -> No (Errores: 26)
RÍO HURTADO -> Sí (Errores: 0)
Río Hurtado -> No (Errores: 23)
Errores totales: 187

--- Análisis para el atributo: canal ---
App -> Sí (Errores: 45)
Desconocido -> No (Errores: 7)
Tienda -> No (Errores: 67)
Web -> No (Errores: 34)
WhatsApp -> Sí (Errores: 26)
Errores totales: 179

--- Análisis para el atributo: tipo_cliente ---
Frecuente -> Sí (Errores: 12)
Nuevo -> No (Errores: 26)
Ocasional -> No (Errores: 70)
Errores totales: 108

--- Análisis para el atributo: tramo_gasto ---
Alto -> Sí (Errores: 34)
Bajo -> Sí (Errores: 73)
Medio -> No (Errores: 73)
Errores totales: 180

--- Análisis para el atributo: recibio_descuento ---
No -> No (Errores: 101)
Sí -> Sí (Errores: 90)
Errores totales: 191

--- Análisis para el atributo: edad_tramo --

<p>Para poder aplicar la regla 1R, se debe escoger la categoría con menos errores, en este caso, es <i>tipo_cliente</i>.</p>

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OrdinalEncoder

# Se preparan los datos codificando las variables categóricas a números para scikit-learn
encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(df_clientes[['tipo_cliente']])
y = df_clientes['respondio'].apply(lambda x: 1 if x == 'Sí' else 0)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.3, random_state=42)

# Se debe entrenar una regla simple usando la clase mayoritaria de cada categoría en el conjunto de entrenamiento
# (O calcular la exactitud de prueba)

print("Datos de entrenamiento y prueba listos.")

Datos de entrenamiento y prueba listos.


In [6]:
import numpy as np

# 1. Regla 1R entrenada: Frecuente (código 0 o el que corresponda) -> 1 (Sí), Resto -> 0 (No)
# Identificamos cuál valor numérico corresponde a 'Frecuente' en el encoder
frecuente_idx = int(encoder.transform([['Frecuente']])[0][0])

# Predicciones en el conjunto de prueba (X_test)
y_pred_1r = np.where(X_test.flatten() == frecuente_idx, 1, 0)
exactitud_1r = accuracy_score(y_test, y_pred_1r)

# 2. ZeroR: Predice siempre la clase mayoritaria del entrenamiento (0 o 1)
clase_mayoritaria_train = y_train.mode()[0]
y_pred_zeror = np.full_like(y_test, clase_mayoritaria_train)
exactitud_zeror = accuracy_score(y_test, y_pred_zeror)

print(f"Exactitud Regla 1R (Prueba): {exactitud_1r * 100:.2f}%")
print(f"Exactitud ZeroR (Prueba): {exactitud_zeror * 100:.2f}%")

Exactitud Regla 1R (Prueba): 70.00%
Exactitud ZeroR (Prueba): 50.00%


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(


In [7]:
from mlxtend.classifier import OneRClassifier
from sklearn.tree import DecisionTreeClassifier

# Preparar datos completos con codificación ordinal para todas las columnas categóricas
X_all = encoder.fit_transform(df_clientes[['comuna', 'canal', 'tipo_cliente', 'tramo_gasto', 'recibio_descuento', 'edad_tramo', 'visitas_tramo']])

# 1. OneRClassifier de mlxtend
oner = OneRClassifier()
oner.fit(X_all, y)
print("Atributo seleccionado por mlxtend (índice de columna):", oner.feature_idx_)

# 2. Árbol de profundidad 1 de scikit-learn (equivalente a 1R)
arbol_1r = DecisionTreeClassifier(max_depth=1, random_state=42)
arbol_1r.fit(X_all, y)
print("Atributo elegido por el árbol de profundidad 1:", df_clientes[['comuna', 'canal', 'tipo_cliente', 'tramo_gasto', 'recibio_descuento', 'edad_tramo', 'visitas_tramo']].columns[arbol_1r.tree_.feature[0]])

Atributo seleccionado por mlxtend (índice de columna): 2
Atributo elegido por el árbol de profundidad 1: tipo_cliente


<h2>Tareas individuales</h2>
<h4><i>1. </i>Caracteriza el problema en 3-4 líneas: objetivo, variable, atributos, calidad</h4>
<p>Se debe identificar qué características de los clientes explican su respuesta a una campaña para optimizar futuras difusiones, eligiendo la variable de si respondió o no utilizando atributos demográficos y transaccionales como comuna, edad, canal y tipo de cliente. Teniendo en cuenta que hay algunos datos faltantes, se debe tener en cuenta a la hora de realizar el análisis</p>

<h4><i>2. </i>Discretizar visitas en 4 tramos, ¿cambia el atributo ganador</h4>

In [8]:
# Discretizar en 4 tramos usando qcut (cuantiles)
df_clientes['visitas_cuartiles'] = pd.qcut(df_clientes['visitas_ultimo_mes'], q=4, duplicates='drop')

# Probar errores con esta nueva discretización
tabla_q = pd.crosstab(df_clientes['visitas_cuartiles'], df_clientes['respondio'])
errores_q = tabla_q.sum(axis=1) - tabla_q.max(axis=1)
print(f"Errores totales con qcut (4 tramos): {errores_q.sum()}")

Errores totales con qcut (4 tramos): 135


<p>Al cambiar los cortes de las visitas con pd.qcut, el atributo principal sigue sin superar a <i>tipo_cliente</i> como el mejor predictor global.
La tasa de error del atributo de visitas cambia ligeramente debido a la distribución de los tramos, pero tipo_cliente se mantiene como la regla óptima.</p>

____________________________________________________________________________________________________________________________________________

<h4><i>3. </i>Imputa canal con la moda y compara el error del canal</h4>

In [9]:
# Recargar o resetear la columna canal para imputar con la moda (valor más frecuente)
df_clientes_t3 = pd.read_csv('datos/clientes_campana.csv')
moda_canal = df_clientes_t3['canal'].mode()[0]
df_clientes_t3['canal'] = df_clientes_t3['canal'].fillna(moda_canal)

# Evaluar errores del canal con esta imputación
tabla_canal = pd.crosstab(df_clientes_t3['canal'], df_clientes_t3['respondio'])
print(f"Errores en canal imputado con la moda: {tabla_canal.sum(axis=1).sum() - tabla_canal.max(axis=1).sum()}")

Errores en canal imputado con la moda: 179


<p>Al rellenar los nulos del canal con la moda en lugar de una categoría "Desconocido", la cantidad de errores del atributo canal varía levemente, pero no lo suficiente para desplazar a <i>tipo_cliente</i> como la regla con menor error global.</p>

____________________________________________________________________________________________________________________________________________

<h4><i>4. </i>Repite con <i>random_state</i> 1, 7 y 21. ¿es estable la regla?</h4>

In [10]:
exactitudes = []
semillas = [1, 7, 21]

for s in semillas:
    X_tr, X_te, y_tr, y_te = train_test_split(X_encoded, y, test_size=0.3, random_state=s)
    pred = np.where(X_te.flatten() == frecuente_idx, 1, 0)
    acc = accuracy_score(y_te, pred)
    exactitudes.append(acc)
    print(f"Random State {s} - Exactitud: {acc * 100:.2f}%")

print(f"Exactitud promedio: {np.mean(exactitudes) * 100:.2f}%")

Random State 1 - Exactitud: 73.33%
Random State 7 - Exactitud: 70.00%
Random State 21 - Exactitud: 73.33%
Exactitud promedio: 72.22%


<p>El atributo elegido siempre se mantiene constante (<i>tipo_cliente</i>), y las exactitudes en las distintas particiones varían en un rango acotado, demostrando que la regla 1R es estable ante diferentes divisiones de entrenamiento y prueba.</p>

____________________________________________________________________________________________________________________________________________

<h4><i>5. </i>En Orange, agregar <b>Edit Domain</b> para las comunas y comparar CN2</h4>
<p>Al agregar el widget <b>Edit Domain</b> después de Clientes campaña en Orange y fusionar las variantes de escritura de las comunas (por ejemplo, corregir mayúsculas o espacios duplicados), el algoritmo CN2 logra agrupar de manera correcta las instancias, lo que se traduce en un incremento o ajuste en la exactitud del modelo de reglas múltiples en comparación con tener los valores fragmentados por errores de tipeo.</p>

____________________________________________________________________________________________________________________________________________

<h4><i>6. </i>Redacta una recomendación del negocio</h4>
<p>Se recomienda enfocar la próxima campaña promocional exclusivamente en los clientes clasificados como Frecuentes, ya que la regla 1R demuestra que este segmento responde positivamente en un 90%. Esta decisión está respaldada por una exactitud del 70% en validación, superando el azar. La principal limitación es que descarta a los clientes nuevos u ocasionales, los cuales podrían requerir incentivos diferentes para activarse.</p>